# NeuroVista — real MONAI BraTS inference (Google Colab)

**Research and education only — not for clinical diagnosis.** This notebook runs the official MONAI `brats_mri_segmentation` v0.5.4 weights. GPU execution is unverified in the NeuroVista repository until this notebook completes successfully on a compatible four-sequence case.

Running the upload cell transfers the four selected MRI files to the active Google Colab runtime. Use only de-identified research data that you are authorized to process in that environment. Do not upload names, medical-record numbers, dates of birth, or other patient-identifying data. Colab is a third-party cloud service; review its current terms and data-handling policy before proceeding.

## 1. Install the model's pinned software stack
The official v0.5.4 metadata specifies PyTorch 2.4.0, MONAI 1.4.0, NumPy 1.24.4, and NiBabel 5.2.1. Restart the runtime if Colab asks after installation.

In [ ]:
%pip install --quiet --upgrade "torch==2.4.0" "monai==1.4.0" "numpy==1.24.4" "nibabel==5.2.1" "huggingface_hub==0.27.1" "psutil==6.1.1"

In [ ]:
import hashlib, json, os, platform, shutil, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

import nibabel as nib
import numpy as np
import psutil
import torch
import monai
from huggingface_hub import hf_hub_download
from monai.inferers import sliding_window_inference
from monai.networks.nets import SegResNet
from monai.transforms import NormalizeIntensity

ram = psutil.virtual_memory()
hardware = {
    'ram_total_gib': round(ram.total / 1024**3, 2),
    'ram_available_gib': round(ram.available / 1024**3, 2),
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_vram_gib': round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if torch.cuda.is_available() else None,
}
print(json.dumps(hardware, indent=2))
print({'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__, 'numpy': np.__version__, 'nibabel': nib.__version__})
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU runtime is required. In Colab choose Runtime > Change runtime type > GPU, then rerun.')

## 2. Download and verify the official model weight
The file is fetched from the versioned official MONAI Hugging Face repository. Its published SHA-256 must match before loading. `torch.load(..., weights_only=True)` avoids executing arbitrary pickle globals.

In [ ]:
MODEL_REPO = 'MONAI/brats_mri_segmentation'
MODEL_VERSION = '0.5.4'
MODEL_FILE = 'models/model.pt'
MODEL_SOURCE = f'https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_VERSION}/{MODEL_FILE}'
EXPECTED_WEIGHT_SHA256 = '860ccb3f1c21c99d0410ad8a1ac4ef6b8fab60cec0a503b0ba42675741a750ae'
WORK = Path('/content/neurovista_inference')
WORK.mkdir(parents=True, exist_ok=True)
weight_path = Path(hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, revision=MODEL_VERSION, local_dir=WORK / 'bundle'))
weight_sha256 = hashlib.sha256(weight_path.read_bytes()).hexdigest()
if weight_sha256 != EXPECTED_WEIGHT_SHA256:
    raise RuntimeError(f'Weight checksum mismatch: {weight_sha256}')
print(json.dumps({'source': MODEL_SOURCE, 'version': MODEL_VERSION, 'sha256': weight_sha256, 'bytes': weight_path.stat().st_size}, indent=2))

## 3. Upload four authorized, de-identified NIfTI volumes
Upload exactly one aligned **T1c/T1ce**, **T1**, **T2**, and **FLAIR** volume from the same case. Files must already be co-registered, use matching matrices/affines, and have 1 mm isotropic spacing as required by this model. The notebook refuses incompatible geometry and never silently aligns scans.

In [ ]:
from google.colab import files
uploaded = files.upload()  # This action uploads selected files to the Colab runtime.
print(f'Uploaded {len(uploaded)} file(s) to this cloud session.')

In [ ]:
# Edit only these four values to match the uploaded filenames. Model channel order is fixed.
INPUTS = {
    't1c': '/content/REPLACE_WITH_T1C.nii.gz',
    't1': '/content/REPLACE_WITH_T1.nii.gz',
    't2': '/content/REPLACE_WITH_T2.nii.gz',
    'flair': '/content/REPLACE_WITH_FLAIR.nii.gz',
}
CHANNEL_ORDER = ['t1c', 't1', 't2', 'flair']
REFERENCE_MODALITY = 't1c'

## 4. Validate geometry and apply documented preprocessing
The bundle uses nonzero, channel-wise intensity normalization. It does not document reorientation or resampling in this inference path, so this notebook requires the documented 1 mm aligned geometry. Because no spatial transform is applied, the prediction grid is already the original T1c reference grid; the export still reasserts and verifies the original shape, affine, units, and header.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

images, arrays, input_manifest = {}, {}, {}
for modality in CHANNEL_ORDER:
    path = Path(INPUTS[modality])
    if not path.is_file() or not (path.name.lower().endswith('.nii') or path.name.lower().endswith('.nii.gz')):
        raise ValueError(f'{modality}: select an existing .nii or .nii.gz file')
    image = nib.load(str(path))
    data = np.asarray(image.dataobj)
    if data.ndim != 3 or not np.issubdtype(data.dtype, np.number) or not np.isfinite(data).all():
        raise ValueError(f'{modality}: expected one finite numeric 3D volume')
    affine = np.asarray(image.affine, dtype=np.float64)
    if not np.isfinite(affine).all() or abs(np.linalg.det(affine[:3, :3])) < 1e-12:
        raise ValueError(f'{modality}: invalid affine')
    spacing = np.asarray(nib.affines.voxel_sizes(affine))
    if not np.allclose(spacing, (1, 1, 1), rtol=0, atol=1e-3):
        raise ValueError(f'{modality}: model requires 1 mm isotropic spacing; received {spacing.tolist()}')
    units = image.header.get_xyzt_units()[0]
    if units != 'mm':
        raise ValueError(f'{modality}: expected millimetre spatial units, received {units!r}')
    images[modality], arrays[modality] = image, data.astype(np.float32, copy=False)
    input_manifest[modality] = {'sha256': sha256_file(path), 'shape': list(data.shape), 'spacing_mm': spacing.tolist(), 'axis_codes': list(nib.aff2axcodes(affine))}

reference = images[REFERENCE_MODALITY]
for modality in CHANNEL_ORDER[1:]:
    if images[modality].shape != reference.shape:
        raise ValueError(f'{modality}: shape does not match T1c reference')
    if not np.allclose(images[modality].affine, reference.affine, rtol=1e-5, atol=1e-4):
        raise ValueError(f'{modality}: affine does not match T1c reference; automatic alignment is disabled')

stacked = np.stack([arrays[name] for name in CHANNEL_ORDER], axis=0)
normalized = NormalizeIntensity(nonzero=True, channel_wise=True)(stacked)
model_input = torch.as_tensor(np.asarray(normalized), dtype=torch.float32).unsqueeze(0)
print(json.dumps({'channel_order': CHANNEL_ORDER, 'tensor_shape': list(model_input.shape), 'reference_affine': reference.affine.tolist(), 'inputs': input_manifest}, indent=2))

## 5. Load verified weights and run real GPU inference
This cell performs real inference. It may fail with an actionable CUDA out-of-memory error on smaller GPUs. Do not report a prediction unless this cell and the export checks both complete.

In [ ]:
device = torch.device('cuda:0')
model = SegResNet(spatial_dims=3, init_filters=16, in_channels=4, out_channels=3, dropout_prob=0.2, blocks_down=(1, 2, 2, 4), blocks_up=(1, 1, 1)).to(device)
checkpoint = torch.load(weight_path, map_location='cpu', weights_only=True)
state_dict = checkpoint.get('model', checkpoint) if isinstance(checkpoint, dict) else checkpoint
model.load_state_dict(state_dict, strict=True)
model.eval()

def predictor(window):
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        return model(window)

torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
with torch.inference_mode():
    logits = sliding_window_inference(model_input.to(device), roi_size=(240, 240, 160), sw_batch_size=1, predictor=predictor, overlap=0.5, device=torch.device('cpu'), sw_device=device)
elapsed_seconds = time.perf_counter() - started
channels = (torch.sigmoid(logits)[0] >= 0.5).cpu().numpy()
if channels.shape != (3, *reference.shape):
    raise RuntimeError(f'Unexpected output shape {channels.shape}')
print({'elapsed_seconds': round(elapsed_seconds, 3), 'peak_gpu_memory_gib': round(torch.cuda.max_memory_allocated() / 1024**3, 3), 'foreground_voxels_tc_wt_et': [int(v.sum()) for v in channels]})

## 6. Restore/reference geometry, export, and package for NeuroVista
The output label map uses BraTS-style values: `0` background, `1` tumour core excluding enhancing override, `2` whole-tumour-only region, and `4` enhancing tumour. TC/WT/ET are nested model outputs, not tumour types. Import the NIfTI locally by loading the matching T1c scan, choosing **AI prediction import**, and selecting this mask. Keep the JSON beside it for provenance review.

In [ ]:
tc, wt, et = channels
label_map = np.zeros(reference.shape, dtype=np.uint8)
label_map[wt] = 2
label_map[tc] = 1
label_map[et] = 4

result_dir = WORK / 'results'
result_dir.mkdir(parents=True, exist_ok=True)
mask_path = result_dir / 'neurovista_ai_prediction.nii.gz'
header = reference.header.copy()
header.set_data_dtype(np.uint8)
prediction_image = nib.Nifti1Image(label_map, reference.affine.copy(), header=header)
nib.save(prediction_image, mask_path)
reloaded = nib.load(str(mask_path))
if reloaded.shape != reference.shape or not np.allclose(reloaded.affine, reference.affine, rtol=0, atol=1e-6):
    raise RuntimeError('Exported prediction did not preserve the original T1c geometry')

provenance = {
    'application': 'NeuroVista Colab inference',
    'research_only': True,
    'gpu_execution_verified_for_this_run': True,
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'model': {'name': 'brats_mri_segmentation', 'version': MODEL_VERSION, 'source': MODEL_SOURCE, 'weight_sha256': weight_sha256, 'license': 'Apache-2.0'},
    'software': {'python': platform.python_version(), 'torch': torch.__version__, 'monai': monai.__version__, 'numpy': np.__version__, 'nibabel': nib.__version__},
    'hardware': hardware,
    'input_contract': {'channel_order': CHANNEL_ORDER, 'reference': REFERENCE_MODALITY, 'inputs': input_manifest},
    'preprocessing': {'spatial_transform': 'none; inputs validated as aligned 1 mm reference geometry', 'intensity': 'NormalizeIntensity(nonzero=True, channel_wise=True)'},
    'inference': {'roi_size': [240, 240, 160], 'sw_batch_size': 1, 'overlap': 0.5, 'threshold': 0.5, 'elapsed_seconds': elapsed_seconds, 'peak_gpu_memory_gib': torch.cuda.max_memory_allocated() / 1024**3},
    'output': {'filename': mask_path.name, 'sha256': sha256_file(mask_path), 'shape': list(reloaded.shape), 'affine': reloaded.affine.tolist(), 'spatial_units': reloaded.header.get_xyzt_units()[0], 'labels': {'0': 'background', '1': 'tumour core', '2': 'whole-tumour-only region', '4': 'enhancing tumour'}, 'provenance': 'AI prediction'},
    'limitations': ['Not for clinical diagnosis', 'Does not classify tumour type', 'Requires expert review'],
}
provenance_path = result_dir / 'neurovista_ai_prediction_provenance.json'
provenance_path.write_text(json.dumps(provenance, indent=2) + '\n', encoding='utf-8')
(result_dir / 'README_IMPORT.txt').write_text('In local NeuroVista: upload the matching T1c scan, choose AI prediction import, then upload neurovista_ai_prediction.nii.gz. Review the provenance JSON.\n', encoding='utf-8')
package_path = WORK / 'neurovista_prediction_package.zip'
with zipfile.ZipFile(package_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for item in result_dir.iterdir(): archive.write(item, arcname=item.name)
print(json.dumps(provenance, indent=2))
print(f'Package ready: {package_path}')
files.download(str(package_path))

## 7. Session cleanup
After confirming the ZIP downloaded, set `CLEAN_UP_NOW = True` and run the cell below. It deletes the selected inputs, weights, predictions, and working files from the active runtime. Then use **Runtime → Disconnect and delete runtime**. Local deletion cannot guarantee the cloud provider has immediately purged every infrastructure-level copy; consult the provider's current retention policy.

In [ ]:
CLEAN_UP_NOW = False
if CLEAN_UP_NOW:
    for path in {Path(value) for value in INPUTS.values()}:
        if path.is_file(): path.unlink()
    shutil.rmtree(WORK, ignore_errors=True)
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print('NeuroVista working files removed from the active runtime. Now disconnect and delete the runtime.')
else:
    print('Cleanup not run. Download results first, then set CLEAN_UP_NOW = True.')